# Train a LLaDA-Inspired BERT Diffusion Prototype in Colab

This notebook fine-tunes a pretrained BERT masked language model with a diffusion-style masking objective and then uses the repo's iterative denoising sampler for generation.

It is designed for Colab-scale experimentation rather than full research reproduction.

## 1. Install dependencies and clone the repo

In [ ]:
!pip -q install -U pip
!pip -q install -U transformers datasets accelerate evaluate

from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/lekkalapudiswetha-work/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM.git"
REPO_DIR = Path("/content/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM")

if not REPO_DIR.exists():
    !git clone {REPO_URL}

%cd /content/NLP_Project_Diffusion-Inspired_Iterative_Refinement_for_SLM
!pip -q install -e .

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 23.5 MB/s eta 0:00:00


## 2. Imports and experiment config

In [ ]:
from dataclasses import dataclass
import math
import random

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

from llada_bert import BertMaskedLMWrapper, DiffusionNoiseScheduler, IterativeDenoisingSampler


@dataclass
class ColabConfig:
    model_name: str = "bert-base-uncased"
    dataset_name: str = "wikitext"
    dataset_config: str = "wikitext-2-raw-v1"
    output_dir: str = "/content/llada_bert_checkpoints"
    max_length: int = 64
    train_examples: int = 4000
    eval_examples: int = 500
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size: int = 8
    learning_rate: float = 5e-5
    num_train_epochs: int = 1.0
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    logging_steps: int = 50
    save_steps: int = 200
    eval_steps: int = 200
    min_mask_ratio: float = 0.15
    max_mask_ratio: float = 0.70
    seed: int = 42


cfg = ColabConfig()

random.seed(cfg.seed)
np.random.seed(cfg.seed)
torch.manual_seed(cfg.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
cfg

device: cuda


ColabConfig(model_name='bert-base-uncased', dataset_name='wikitext', dataset_config='wikitext-2-raw-v1', output_dir='/content/llada_bert_checkpoints', max_length=64, train_examples=4000, eval_examples=500, per_device_train_batch_size=8, per_device_eval_batch_size=8, learning_rate=5e-05, num_train_epochs=1.0, weight_decay=0.01, warmup_ratio=0.1, logging_steps=50, save_steps=200, eval_steps=200, min_mask_ratio=0.15, max_mask_ratio=0.7, seed=42)

## 3. Load a small text dataset

In [ ]:
raw = load_dataset(cfg.dataset_name, cfg.dataset_config)

train_texts = [x["text"] for x in raw["train"] if x["text"].strip()]
eval_texts = [x["text"] for x in raw["validation"] if x["text"].strip()]

train_texts = train_texts[: cfg.train_examples]
eval_texts = eval_texts[: cfg.eval_examples]

print("train examples:", len(train_texts))
print("eval examples:", len(eval_texts))
print("sample:", train_texts[0][:200])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

train examples: 4000
eval examples: 500
sample:  = Valkyria Chronicles III = 



## 4. Tokenize examples

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=cfg.max_length,
        padding=False,
    )

train_ds = raw["train"].filter(lambda ex: bool(ex["text"].strip())).select(range(len(train_texts)))
eval_ds = raw["validation"].filter(lambda ex: bool(ex["text"].strip())).select(range(len(eval_texts)))

train_ds = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
eval_ds = eval_ds.map(tokenize_batch, batched=True, remove_columns=eval_ds.column_names)

train_ds[0]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Filter:   0%|          | 0/36718 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

{'input_ids': [101, 1027, 11748, 4801, 4360, 11906, 3523, 1027, 102],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

## 5. Diffusion-style masking collator

For each sequence we sample a noise level, mask that fraction of non-special tokens, and train the model to reconstruct the masked tokens.

In [ ]:
class DiffusionMaskingCollator:
    def __init__(self, tokenizer, min_mask_ratio=0.15, max_mask_ratio=0.70):
        self.tokenizer = tokenizer
        self.min_mask_ratio = min_mask_ratio
        self.max_mask_ratio = max_mask_ratio
        self.pad_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

    def __call__(self, examples):
        batch = self.pad_collator(examples)
        input_ids = batch["input_ids"].clone()
        attention_mask = batch["attention_mask"]
        labels = torch.full_like(input_ids, fill_value=-100)

        special_tokens_mask = torch.tensor(
            [
                self.tokenizer.get_special_tokens_mask(row.tolist(), already_has_special_tokens=True)
                for row in input_ids
            ],
            dtype=torch.bool,
        )

        valid_token_mask = (attention_mask == 1) & (~special_tokens_mask)

        for row_idx in range(input_ids.size(0)):
            candidate_positions = torch.where(valid_token_mask[row_idx])[0]
            if len(candidate_positions) == 0:
                continue

            noise_level = random.uniform(self.min_mask_ratio, self.max_mask_ratio)
            num_to_mask = max(1, int(math.ceil(noise_level * len(candidate_positions))))
            selected_order = torch.randperm(len(candidate_positions))[:num_to_mask]
            mask_positions = candidate_positions[selected_order]

            labels[row_idx, mask_positions] = input_ids[row_idx, mask_positions]
            input_ids[row_idx, mask_positions] = self.tokenizer.mask_token_id

        batch["input_ids"] = input_ids
        batch["labels"] = labels
        return batch


collator = DiffusionMaskingCollator(
    tokenizer=tokenizer,
    min_mask_ratio=cfg.min_mask_ratio,
    max_mask_ratio=cfg.max_mask_ratio,
)

sample_batch = collator([train_ds[i] for i in range(2)])
sample_batch.keys()

KeysView({'input_ids': tensor([[  101,  1027,   103,  4801,   103, 11906,   103,   103,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0],
        [  101, 12411,  5558,  2053,   103,   103,  4360,  1017,  1024,  4895,
          2890, 27108,   103,   103,  1006,  2887,  1024,  1856,  1806,  1671,
         30222, 30218, 30259,   103, 30255, 30258, 30219,  2509,  1010,  5507,
           103, 11748,  4801,   103,  1997,  1996, 11686,  1017,  1007,  1010,
           103,   103,  2000,  2004, 11748,  4801,  4360, 11906,  3523,  2648,
          2900,  1010,  2003,  1037,   103,  2535,  1030,  1011,   103

## 6. Initialize the model

In [ ]:
model = AutoModelForMaskedLM.from_pretrained(cfg.model_name)
model.to(device)
print(f"Loaded {cfg.model_name}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded bert-base-uncased


## 7. Fine-tune with the diffusion-style objective

In [ ]:
training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.num_train_epochs,
    weight_decay=cfg.weight_decay,
    warmup_steps=100,
    logging_steps=cfg.logging_steps,
    eval_steps=cfg.eval_steps,
    save_steps=cfg.save_steps,
    eval_strategy="steps",
    save_strategy="steps",
    report_to="none",
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
)

trainer.train()

Step,Training Loss,Validation Loss
200,4.083862,3.920176
400,3.809430,3.805989
500,3.715085,3.775350


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=3.998038299560547, metrics={'train_runtime': 32.8938, 'train_samples_per_second': 121.603, 'train_steps_per_second': 15.2, 'total_flos': 131602406400000.0, 'train_loss': 3.998038299560547, 'epoch': 1.0})

## 8. Save the fine-tuned checkpoint

In [ ]:
SAVE_DIR = "/content/llada_bert_finetuned"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("saved to", SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to /content/llada_bert_finetuned


## 9. Run iterative denoising generation with the fine-tuned model

In [ ]:
wrapper = BertMaskedLMWrapper(model_name=SAVE_DIR, device=device)
scheduler = DiffusionNoiseScheduler(total_steps=12, base_threshold=0.85)
sampler = IterativeDenoisingSampler(
    model=wrapper,
    scheduler=scheduler,
    threshold=0.85,
    top_k=25,
)

result = sampler.sample(
    prompt="language models can improve reasoning by",
    batch_size=2,
    sequence_length=24,
    steps=12,
    temperature=0.9,
    remask_strategy="low_confidence",
)

for idx, text in enumerate(result.texts):
    print(f"sample {idx + 1}: {text}")

result.logger.as_rows()[:5]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

sample 1: language models can improve reasoning by using the rules of grammar, grammar, and syntax. grammar models can tox
sample 2: language models can improve reasoning by using the language model. the results of the language model and the thesaurus,


[{'step': 0,
  'masked_tokens': 32,
  'changed_tokens': 0,
  'mean_confidence': 0.3388,
  'accepted_tokens': 0},
 {'step': 1,
  'masked_tokens': 28,
  'changed_tokens': 4,
  'mean_confidence': 0.3583,
  'accepted_tokens': 1},
 {'step': 2,
  'masked_tokens': 26,
  'changed_tokens': 2,
  'mean_confidence': 0.4364,
  'accepted_tokens': 0},
 {'step': 3,
  'masked_tokens': 22,
  'changed_tokens': 4,
  'mean_confidence': 0.4692,
  'accepted_tokens': 0},
 {'step': 4,
  'masked_tokens': 20,
  'changed_tokens': 2,
  'mean_confidence': 0.5501,
  'accepted_tokens': 0}]

## 10. Optional next experiments

- increase `num_train_epochs`
- swap `wikitext-2-raw-v1` for a domain-specific text dataset
- compare different `min_mask_ratio` and `max_mask_ratio` settings
- try larger sequence lengths on a GPU runtime
- save checkpoints to Google Drive